# v4n rebuilt engine, e9-sub1 (our code, MIT)
test.parquet -> our v4r engine (cft_hoR_a bank) -> ranker_v0 -> top-25 unique by score key -> submission.csv.
Thin wrapper around `predict.py` from `shishiradhikari11/casmi-v4n-engine-code`. Inputs: competition data;
our tables (train structures + COCONUT, CC BY 4.0); our models; RDKit 2026.03.3 wheel (BSD). CPU only, no internet.

In [ ]:
import os, sys, glob, shutil, subprocess, time, json
T0 = time.time()
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'

def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

# RDKit 2026.03.3 (competition metric version)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = sorted((w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w),
             key=lambda w: 'rdkit-2026.3.3-' not in w)
print('rdkit wheel:', whl)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
v = subprocess.run([sys.executable, '-c', 'import rdkit; print(rdkit.__version__)'], capture_output=True, text=True).stdout.strip()
print('rdkit', v)
assert v == '2026.03.3', v

# Rebuild the repo layout the code expects (engine/model.py finds cft_model at engine/../../train_cft)
CODE = os.path.dirname(find('predict.py'))
ROOT = '/kaggle/working/code'
dirs = {'engine': f'{ROOT}/research/v4n_rebuild/engine', 'sim': f'{ROOT}/research/v4n_rebuild/sim',
        'cft': f'{ROOT}/research/train_cft'}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)
for f in glob.glob(f'{CODE}/*.py'):
    b = os.path.basename(f)
    dst = dirs['sim'] if b in ('common.py', 'simulate.py', 'predict.py') else dirs['cft'] if b.startswith('cft_') else dirs['engine']
    shutil.copy(f, dst)

SRC_TABLES = os.path.dirname(find('pool_fp.npy'))
TABLES = '/kaggle/working/tables'   # v3: local disk; random-access mmaps on /kaggle/input are slow
t = time.time(); shutil.copytree(SRC_TABLES, TABLES, dirs_exist_ok=True)
print(f'tables copied to local disk in {time.time() - t:.0f}s', flush=True)
CKPT = find('cft_hoR_a.pt')
RANKER = find('ranker_v0.pkl')
TEST = find('test.parquet')
print('CODE', CODE, '\nTABLES', TABLES, '\nCKPT', CKPT, '\nRANKER', RANKER, '\nTEST', TEST)
print('cpus', os.cpu_count())

In [ ]:
env = dict(os.environ, V4R_TABLES=TABLES, V4R_SIM_OUT='/kaggle/working/sim')
WORKERS = max(1, min(4, os.cpu_count() or 1))
cmd = [sys.executable, '-u', f"{dirs['sim']}/predict.py", '--test', TEST, '--tables', TABLES, '--ckpt', CKPT,
       '--ranker', RANKER, '--out', '/kaggle/working/submission.csv', '--workers', str(WORKERS), '--key-check', '200']
# v3: smoke run first (3 molecules, 1 worker, 20 min cap) so a hang fails fast with a visible log
smoke = cmd[:cmd.index('--out')] + ['--out', '/kaggle/working/smoke.csv', '--workers', '1', '--limit', '3']
print(' '.join(smoke), flush=True); t = time.time()
p = subprocess.run(smoke, env=env, cwd='/kaggle/working', timeout=1200)
assert p.returncode == 0, p.returncode
print(f'smoke OK in {time.time() - t:.0f}s', flush=True)
print(' '.join(cmd), flush=True)
p = subprocess.run(cmd, env=env, cwd='/kaggle/working', timeout=6 * 3600)
assert p.returncode == 0, p.returncode

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
test = pd.read_parquet(TEST, columns=['molecule_id'])
n = sub.smiles.str.split(';').map(lambda x: len(set(x)))
print('rows', len(sub), 'test molecules', test.molecule_id.nunique())
print('rows with 25 unique SMILES', int((n == 25).sum()))
print('ids match', set(sub.molecule_id) == set(test.molecule_id))
rep = json.load(open('/kaggle/working/submission.report.json'))
print(json.dumps({k: rep.get(k) for k in ('rdkit', 'key_check', 'rows_25_valid', 'rows_padded', 'errors',
                                         'wall_s_per_mol', 'worker_s_per_mol')}, indent=1))
shutil.rmtree(TABLES, ignore_errors=True); shutil.rmtree('/kaggle/working/code', ignore_errors=True)  # keep output small
print(f'total {time.time() - T0:.0f}s')
sub.head()